# Notebook 5a: Computing Inter-Brain Synchrony (INS) for EEG
### Amplitude-based vs. Phase-based Metrics, EEG vs. fNIRS Considerations, and Linear vs. Non-Linear Approaches

In this notebook we:
- Load cleaned EEG data from the first notebook using pickle.
- Define frequency bands and extract the sampling rate.
- Prepare the data for connectivity analysis.
- Compute several INS metrics using functions from `hypyp.analyses`:
  - **Amplitude-based metrics:** envelope correlation, power correlation.
  - **Phase-based metrics:** PLV, circular correlation (ccorr), coherence, imaginary coherence, pli, and wpli.
- Visualize one example connectivity matrix.
- Discuss EEG vs. fNIRS considerations and linear vs. non-linear approaches.

Note: Although here we use EEG data, keep in mind that fNIRS signals are slower and predominantly amplitude-based. Thus, envelope‐based measures might be more appropriate for fNIRS.

In [ ]:
import os
import numpy as np
import mne
import matplotlib.pyplot as plt
import pickle

# Import the connectivity functions from analyses.py (part of hypyp)
import hypyp.analyses as analyses

# Import basic visualization functions (for example, a 2D topomap function)
import hypyp.viz as viz

# For reproducibility
np.random.seed(42)

print("Libraries imported successfully.")

## 1. Load Participant Data

Here we load two EEG epochs files (one per participant) in .fif format.
Make sure the file names match your local files.

In [ ]:
# Load the preprocessed data saved in the first notebook
try:
    with open('./data/hyperscanning_data.pkl', 'rb') as f:
        saved_data = pickle.load(f)
    
    # Extract the cleaned epochs from the saved data
    epo1 = saved_data['epo1_clean']
    epo2 = saved_data['epo2_clean']
    
    print("Successfully loaded preprocessed data from pickle file.")
    print("Participant 1 Epochs:", epo1)
    print("Participant 2 Epochs:", epo2)
    
except FileNotFoundError:
    print("Pickle file not found. Falling back to loading .fif files.")
    
    # Update file paths as needed
    file1 = "./data/participant1-epo.fif"
    file2 = "./data/participant2-epo.fif"

    # Load the epochs using MNE (assumed to be preprocessed and stored as epochs)
    epo1 = mne.read_epochs(file1, preload=True)
    epo2 = mne.read_epochs(file2, preload=True)
    
    print("Participant 1 Epochs:", epo1)
    print("Participant 2 Epochs:", epo2)

# Ensure epoch counts are equal for both participants
mne.epochs.equalize_epoch_counts([epo1, epo2])

## 3. Define Frequency Bands and Prepare Data

We define frequency bands of interest. For this example, we use a narrow frequency range (e.g., 8–12 Hz for the alpha band).
Then we prepare a data array for the two participants.

In [ ]:
# Get the sampling rate (assume both epochs share the same sfreq)
sfreq = epo1.info['sfreq']

# Define frequency bands as a dictionary (you may add additional bands if desired)
freq_bands = {'alpha': [8, 12]}

# Prepare a 4D data array: shape (2, n_epochs, n_channels, n_times)
dyad = np.array([epo1.get_data(copy=True), epo2.get_data(copy=True)])
print("Data shape (participants, epochs, channels, times):", dyad.shape)

## 4. Compute INS Using Different Metrics

We now compute the connectivity (INS) for our dyad using the `pair_connectivity` function from analyses.py.
We loop through several methods. Here we separate them into two groups:

**Amplitude-based metrics (Linear):**
- *Envelope Correlation* (`'envelope_corr'`)
- *Power Correlation* (`'pow_corr'`)

**Phase-based metrics (Which may capture non-linear aspects):**
- *Phase Locking Value (PLV)* (`'plv'`)
- *Circular Correlation* (`'ccorr'`)
- *Coherence* (`'coh'`)
- *Imaginary Coherence* (`'imaginary_coh'`)
- *Phase Lag Index (PLI)* (`'pli'`)
- *Weighted PLI (wPLI)* (`'wpli'`)

For each, we compute a connectivity matrix. Note that the returned connectivity matrix has shape (n_freq, 2*n_channels, 2*n_channels). For visualization we’ll select the first frequency bin.

In [ ]:
# Define a helper function to compute and return connectivity for a given metric
def compute_connectivity(data, metric):
    
    con = analyses.pair_connectivity(
        data, 
        sampling_rate=sfreq, 
        frequencies=freq_bands, 
        mode=metric, 
        epochs_average=True,
    )
    print(f"Connectivity matrix for {metric} computed with shape: {con.shape}")
    return con

# Amplitude-based metrics
print("\nComputing amplitude-based metrics:")
conn_env = compute_connectivity(dyad, 'envelope_corr')
conn_pow = compute_connectivity(dyad, 'pow_corr')

# Phase-based metrics
print("\nComputing phase-based metrics:")
conn_plv = compute_connectivity(dyad, 'plv')
conn_ccorr = compute_connectivity(dyad, 'ccorr')
conn_coh = compute_connectivity(dyad, 'coh')
conn_imag_coh = compute_connectivity(dyad, 'imaginary_coh')
conn_pli = compute_connectivity(dyad, 'pli')
conn_wpli = compute_connectivity(dyad, 'wpli')

## 5. Visualization of a Connectivity Matrix

We now visualize one example connectivity matrix. Here we use the 2D topographic visualization from `hypyp.viz`.
For instance, we plot the connectivity matrix computed using circular correlation (`conn_ccorr`).
Since the connectivity matrix has shape (n_freq, 2*n_channels, 2*n_channels), we select the first frequency bin.

In [ ]:
# Select the first frequency bin from the connectivity matrix (assuming 1 frequency bin for alpha)
C_ccorr = conn_wpli[0,:31,31:]  # shape: (2*n_channels, 2*n_channels)

# Visualize using a provided visualization function (e.g., 2D topomap for inter-brain connectivity)
# Note: This function is part of hypyp.viz; adjust parameters if necessary.
viz.viz_2D_topomap_inter(epo1, epo2, C_ccorr, threshold='auto', steps=10, lab=True)

## 6. Discussion

- **Amplitude-based vs. Phase-based Metrics:**  
  Amplitude-based measures (such as envelope and power correlations) are typically linear and may capture slow changes in signal amplitude, which can be especially relevant for fNIRS data. Phase-based measures (e.g., PLV, circular correlation, coherence) capture the phase relationship between signals and may reveal more subtle temporal synchrony patterns. Some phase-based measures (e.g., PLI and wPLI) are designed to mitigate the effects of volume conduction.

- **Linear vs. Non-Linear Approaches:**  
  Linear metrics (like envelope and power correlations) assume linear relationships between signals. In contrast, some phase-based metrics can capture non-linear dynamics. The choice of metric may affect sensitivity to true inter-brain interactions versus spurious correlations.

- **EEG vs. fNIRS Considerations:**  
  EEG signals have high temporal resolution and are often used with phase-based metrics. fNIRS signals, which are hemodynamic in nature, are slower and often analyzed using amplitude-based metrics. When analyzing fNIRS data, consider using envelope or power correlation methods, and be mindful of the different signal characteristics.

By comparing results across these methods, one can gain a better understanding of the underlying inter-brain synchrony and the impact of methodological choices on the observed metrics.

## Conclusion

In this notebook, we have:
- Loaded two EEG participant files.
- Prepared the data and defined frequency bands.
- Computed inter-brain connectivity using several metrics—grouped into amplitude-based and phase-based measures.
- Visualized an example connectivity matrix.
- Discussed EEG versus fNIRS considerations and linear versus non-linear approaches.

These steps provide a comprehensive framework for computing INS and understanding how different metrics capture various aspects of inter-brain synchrony.